In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2], [0, 3],
        [1, 0], [1, 1], [1, 2], [1, 3]]
n_edge = [(0, 1), (1, 2), (2, 3), 
          (4, 5), (5, 6), (6, 7),
          (0, 4), (1, 5), (2, 6), (3, 7),
          (1, 4), (2, 5), (3, 6)]
triArea = 1

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [1] * 8

In [ ]:
fuseMarkers[2] = 0
fuseMarkers[6] = 0

In [ ]:
fuseMarkers

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 1, epsilon = 1e-5)

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
import time, vis
# When doing gradient validation, need to disable tension field theory
ipu.sheet.setUseTensionFieldEnergy(False)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations


In [ ]:
ipu.numVars()

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
fixedVars, hessianShift = [], 1e-6

In [ ]:
benchmark.reset()

opts.niter = 10
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(False)

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
from periodic_simulation_setup import *

In [ ]:
useTFT = False
disableFusedRegionTFT = False

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, np.array(fuseMarkers) == 1, useTFT, disableFusedRegionTFT)

In [ ]:
np.set_printoptions(precision=4, suppress=True)

In [ ]:
az_ipu.getVars()

In [ ]:
ipu.getVars()

In [ ]:
import fd_validation

In [ ]:
A = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.getMidSurfaceToPeriodicPatchMapTranspose_all_vars())

In [ ]:
A[3:-2, 3:-2][2::3, 2::3]

In [ ]:
la.norm(la.inv(A.T) @ az_ipu.ipu.getVars() - az_ipu.getVars())

In [ ]:
la.norm((A.T) @ az_ipu.getVars() - az_ipu.ipu.getVars())

In [ ]:
fd_validation.gradConvergencePlot(az_ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(az_ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})